In [ ]:
%matplotlib inline
import json
import os
import random

import numpy as np
import pandas
import pylab as plt
import structuretoolkit as stk
from ase.build import bulk
from ase.data import reference_states, atomic_numbers
from executorlib import SingleNodeExecutor

from interfacemethod import (
    minimize_structure_positions,
    minimize_structure_volume,
    check_diamond,
    analyse_minimized_structure,
    run_npt_step,
    bisection_step,
    initialise_iterators,
    round_temperature_next,
    npt_solid,
    npt_liquid,
    remove_selective_dynamics,
    get_strain_lst,
    run_strain_point,
    plot_solid_liquid_ratio,
    ratio_selection,
    plot_equilibration,
    check_for_holes,
    plot_melting_point_prediction,
    validate_convergence,
)

In [ ]:
project_path = "melting"

In [ ]:
input_file = 'input.json'
output_file = 'output.json'

In [ ]:
lmp_command = "lmp -in lmp.in"

In [ ]:
max_workers = 4

In [ ]:
if os.path.exists(input_file) and os.stat(input_file).st_size != 0:
    with open(input_file, 'r') as f:
        input_dict = json.load(f)
else:
    input_dict = {
        "config": [
            "pair_style morse 9.97749\n",
            "pair_coeff * * 0.4174 1.3885 2.845\n"
        ],
        "species": ["Al"],
        "element": "Al"
    }

In [ ]:
input_dict

In [ ]:
input_dict["config"][1] = input_dict["config"][1].replace(input_dict["filename"], os.path.abspath(input_dict["filename"]))
input_dict

In [ ]:
pot_dict = input_dict.copy()

In [ ]:
if 'model' not in pot_dict.keys():
    pot_dict['model'] = 'Lammps'
if 'name' not in pot_dict.keys():
    pot_dict['name'] = 'CustomPotential'
if 'filename' not in pot_dict.keys():
    pot_path = []
else:
    pot_path = [os.path.abspath(pot_dict['filename'])]
potential = pandas.DataFrame({'Config': [pot_dict['config']],
                             'Filename': [pot_path],
                             'Model': [pot_dict['model']],
                             'Name': [pot_dict['name']],
                             'Species': [pot_dict['species']]
                             })

In [ ]:
project_parameter = {
    'run_time_steps': 50000,
    'nvt_run_time_steps': 10000,
    'nve_run_time_steps': 10000,
    'temperature_left': 0,
    'temperature_right': 1000,
    'strain_run_time_steps': 1000,
    'convergence_criterion': 1,
    'potential': potential,
    'cpu_cores': 1,
    'enable_h5md': False,
    'points': 21,
    'boundary_value': 0.25,
    'ratio_boundary': 0.25,
    'timestep_lst': [2, 2, 1],
    'fit_range_lst': [0.05, 0.01, 0.01],
    'nve_run_time_steps_lst': [25000, 20000, 50000],
    'number_of_atoms': 8000,
    'seed': 56570,
}

In [ ]:
for k in input_dict.keys():
    project_parameter[k] = input_dict[k]

In [ ]:
if 'crystalstructure' not in project_parameter.keys():
    project_parameter['crystalstructure'] = reference_states[atomic_numbers[project_parameter['element']]]['symmetry']

In [ ]:
if 'seed' not in project_parameter.keys():
    project_parameter['seed'] = random.randint(0,99999)
project_parameter['seed']

In [ ]:
# Values from a previous calculation can be inserted here to reproduce the results
step_dict = {}
if os.path.exists(output_file):
    with open(output_file, 'r') as f:
        step_dict_str = json.load(f)
    for k,v in step_dict_str.items():
        step_dict[int(k)] = v
step_dict

# From here on the notebook is automated - no change required !

In [ ]:
step_count = 0
temperature_next = None
enable_iteration = True
convergence_goal_achieved = False

In [ ]:
if 'lattice_constant' in project_parameter.keys():
    a = project_parameter['lattice_constant']
else:
    a = None
if project_parameter['crystalstructure'] == 'hcp':
    basis = bulk(name=project_parameter['element'], crystalstructure=project_parameter['crystalstructure'].lower(), a=a, orthorhombic=True)
else:
    basis = bulk(name=project_parameter['element'], crystalstructure=project_parameter['crystalstructure'].lower(), a=a, cubic=True)
basis_lst = [basis.repeat([i, i, i]) for i in range(5,30)]
basis = basis_lst[np.argmin([np.abs(len(b)-project_parameter['number_of_atoms']/2) for b in basis_lst])]

In [ ]:
stk.analyse.get_adaptive_cna_descriptors(basis, ovito_compatibility=True)

In [ ]:
stk.visualize.plot3d(basis)

In [ ]:
timestep_iter, fit_range_iter, nve_run_time_steps_iter = initialise_iterators(project_parameter)
timestep = next(timestep_iter)
fit_range = next(fit_range_iter)
nve_run_time_steps = next(nve_run_time_steps_iter)

# Step 1: set up the solid sample and roughly estimate a melting point

In [ ]:
with SingleNodeExecutor(cache_directory="cache", max_workers=max_workers) as exe:
    basis_minimize_pos = exe.submit(
        minimize_structure_positions,
        structure=basis, 
        potential=project_parameter["potential"], 
        project_path=project_path,
        lmp_command=lmp_command,
    )
    basis_minimize_vol = exe.submit(
        minimize_structure_volume,
        structure=basis_minimize_pos, 
        potential=project_parameter["potential"], 
        project_path=project_path,
        lmp_command=lmp_command,
    ).result()

In [ ]:
diamond_flag = check_diamond(structure=basis_minimize_vol)
diamond_flag

In [ ]:
(
    structure_after_minimization,
    key_max,
    number_of_atoms,
    distribution_initial_half,
    _,
) = analyse_minimized_structure(structure=basis_minimize_vol)
structure_after_minimization, key_max, number_of_atoms, distribution_initial_half

In [ ]:
def estimate_melting_temperature_using_bisection_CNA(
    structure,
    potential_dataframe,
    project_path,
    number_of_atoms,
    key_max,
    distribution_initial_half,
    diamond_flag,
    lmp_command,
    max_workers=4,
    temperature_left=0,
    temperature_right=1000,
    run=10000,
    seed=None,
):
    """
    Estimate the melting temperature by bisecting a temperature bracket with NPT MD + CNA:

    1. Run NPT MD at the upper bound of the bracket.
    2. Classify both bracket endpoints as solid- or liquid-dominated (CNA) and narrow the
       bracket towards the phase boundary, running one new NPT MD simulation per step.
    3. Repeat until the bracket is narrower than 10 K.
    """
    structure_left = structure

    # Step 1c-i: NPT MD at the initial upper temperature bound
    structure_right = run_npt_step(
        structure=structure,
        temperature=temperature_right,
        seed=seed,
        potential=potential_dataframe,
        run_time_steps=run,
        project_path=project_path,
        lmp_command=lmp_command,
    )
    temperature_step = temperature_right - temperature_left

    # Step 1c-ii: bisect the bracket until it is narrower than 10 K.
    # Each iteration decides -- from the CNA analysis of the *previous* iteration's structures --
    # which single new NPT MD run to launch next, so unlike the strain scan in Step 2 this loop
    # is inherently sequential and cannot be handed to executor.map(). The parallelizable unit of
    # work at this stage of the workflow is a whole bisection search, e.g. running this function
    # for several elements, potentials or initial seeds concurrently.
    with SingleNodeExecutor(cache_directory="cache", max_workers=max_workers) as exe:
        while temperature_step > 10:
            (
                structure_left,
                structure_right,
                temperature_left,
                temperature_right,
            ) = exe.submit(
                bisection_step,
                number_of_atoms=number_of_atoms,
                key_max=key_max,
                structure_left=structure_left,
                structure_right=structure_right,
                potential=potential_dataframe,
                temperature_left=temperature_left,
                temperature_right=temperature_right,
                distribution_initial_half=distribution_initial_half,
                structure_after_minimization=structure,
                run_time_steps=run,
                seed=seed,
                diamond_flag=diamond_flag,
                project_path=project_path,
                lmp_command=lmp_command,
            ).result()
            temperature_step = temperature_right - temperature_left

    return int(round(temperature_left))

**Why this loop is not parallelized.** Each bisection step above launches exactly one new NPT MD
run, and which temperature it runs at is decided by the CNA classification of the *previous*
step's structures -- iteration *N* cannot start before iteration *N-1* finishes and is analysed.
That is different from the strain scan in Step 2, where every strain point starts from the same
structure and all points can be dispatched at once.

If you want to use extra compute here, the independent unit of work is a whole bisection search,
not a single step within one: for example, estimating the melting point for several elements,
potentials, or random seeds at once is embarrassingly parallel across
`estimate_melting_temperature_using_bisection_CNA()` calls, and that outer loop could be handed to
a `ProcessPoolExecutor`/`executorlib.Executor` the same way the strain scan was.

In [ ]:
temperature_next = estimate_melting_temperature_using_bisection_CNA(
    structure=structure_after_minimization,
    potential_dataframe=project_parameter["potential"],
    project_path=project_path,
    number_of_atoms=number_of_atoms,
    key_max=key_max,
    distribution_initial_half=distribution_initial_half,
    diamond_flag=diamond_flag,
    temperature_left=project_parameter["temperature_left"],
    temperature_right=project_parameter["temperature_right"],
    run=project_parameter["strain_run_time_steps"],
    seed=project_parameter['seed'],
    lmp_command=lmp_command,
    max_workers=max_workers,
)
temperature_next

In [ ]:
len(structure_after_minimization)

In [ ]:
temperature_next   # +/- 100K

In [ ]:
if step_count in step_dict.keys():
    temperature_next = step_dict[step_count]['temperature_next']
else:
    step_dict[step_count]= {'temperature_next': temperature_next}

In [ ]:
step_dict

# Step 2: set up the interface structure

In [ ]:
temperature_next = round_temperature_next(temperature_next)
temperature_next

In [ ]:
def run_interface_iteration(
    basis,
    temperature_next,
    project_parameter,
    project_path,
    timestep,
    nve_run_time_steps,
    fit_range,
    center,
    lmp_command,
    max_workers=4,
    debug_plot=True,
):
    """
    Run one iteration of the solid-liquid interface method and predict the next melting
    temperature estimate:

    1. equilibrate the solid at the target temperature (NPT)
    2. partially melt one half of the cell and re-equilibrate to build a solid-liquid interface (NPT)
    3. scan a series of strains around the current lattice parameter (NVT + NVE)
    4. classify each strain point as solid- or liquid-dominated and keep the coexistence region
    5. fit pressure and temperature vs. strain to predict the next melting temperature estimate
    """
    temperature_next = round_temperature_next(temperature_next)

    with SingleNodeExecutor(cache_directory="cache", max_workers=max_workers) as exe:

        # Step 2a: equilibrate the solid at the target temperature (NPT)
        structure_npt_solid = exe.submit(
            npt_solid,
            lmp_command=lmp_command,
            temperature=temperature_next,
            basis=basis,
            project_parameter=project_parameter,
            project_path=project_path,
            timestep=timestep,
        ).result()
        structure_npt_solid.set_velocities(np.zeros_like(structure_npt_solid.get_velocities()))

        # Step 2b: partially melt one half and re-equilibrate to build a solid-liquid interface (NPT)
        structure_npt_liquid_low = exe.submit(
            npt_liquid,
            lmp_command=lmp_command,
            temperature_solid=temperature_next,
            temperature_liquid=temperature_next + 1000,
            basis=structure_npt_solid,
            project_parameter=project_parameter,
            project_path=project_path,
            timestep=timestep,
        ).result()
        basis_relative = remove_selective_dynamics(structure_npt_liquid_low)

        # Step 2c: scan a series of strains around the current lattice parameter (NVT + NVE).
        # Every strain point is independent of every other -- this is the parallelization point of
        # the workflow. See the note below on replacing this `for` loop with a parallel executor.
        strain_lst = get_strain_lst(
            fit_range=fit_range, points=project_parameter["points"], center=center,
        )
        strain_futures = []
        for strain in strain_lst:
            strain_futures.append(
                exe.submit(
                    run_strain_point,
                    strain=strain,
                    lmp_command=lmp_command,
                    basis_relative=basis_relative,
                    temperature_next=temperature_next,
                    nve_run_time_steps=nve_run_time_steps,
                    project_parameter=project_parameter,
                    project_path=project_path,
                    timestep=timestep,
                )
            )

        strain_results = [future.result() for future in strain_futures]

    # Step 2d: classify each strain point as solid- or liquid-dominated
    ratio_lst = plot_solid_liquid_ratio(
        strain_results=strain_results,
        project_parameter=project_parameter,
        debug_plot=debug_plot,
    )
    selected_results, sl_flag = ratio_selection(
        strain_results=strain_results,
        ratio_lst=ratio_lst,
        ratio_boundary=project_parameter["ratio_boundary"],
        debug_plot=debug_plot,
    )

    # Step 2e: fit pressure and temperature vs. strain to predict the next melting temperature
    if len(selected_results) > 2:
        plot_equilibration(strain_results=strain_results, debug_plot=debug_plot)
        ind = check_for_holes(strain_results=selected_results)
        selected_results = [r for r, keep in zip(selected_results, ind) if keep]
        (
            temperature_next,
            temperature_mean,
            temperature_left,
            temperature_right,
        ) = plot_melting_point_prediction(
            strain_results=selected_results,
            boundary_value=project_parameter["boundary_value"],
            debug_plot=True,
        )
    elif sl_flag < 0:
        temperature_next, temperature_mean, temperature_left, temperature_right = (
            temperature_next * 0.90, 0.0, 0.0, 0.0,
        )
    else:
        temperature_next, temperature_mean, temperature_left, temperature_right = (
            temperature_next * 1.10, 0.0, 0.0, 0.0,
        )

    strain_result_lst = [r.strain for r in selected_results]
    pressure_result_lst = [r.pressure for r in selected_results]
    return (
        temperature_next,
        temperature_mean,
        temperature_left,
        temperature_right,
        strain_result_lst,
        pressure_result_lst,
    )

In [ ]:
temperature_next, temperature_mean, temperature_left, temperature_right, strain_result_lst, pressure_result_lst = run_interface_iteration(
    basis=basis_minimize_vol.repeat([1,1,2]),
    temperature_next=temperature_next,
    project_parameter=project_parameter,
    project_path=project_path,
    timestep=timestep,
    nve_run_time_steps=nve_run_time_steps,
    fit_range=fit_range,
    lmp_command=lmp_command,
    center=None,
    debug_plot=True
)

In [ ]:
temperature_next, temperature_mean, temperature_left, temperature_right

# Step 3: run NVT and NVE MD

In [ ]:
output = validate_convergence(
    temperature_left=temperature_left,
    temperature_next=temperature_next,
    temperature_right=temperature_right,
    enable_iteration=enable_iteration,
    timestep_iter=timestep_iter,
    timestep_lst=project_parameter['timestep_lst'],
    timestep=timestep,
    fit_range_iter=fit_range_iter,
    fit_range_lst=project_parameter['fit_range_lst'],
    fit_range=fit_range,
    nve_run_time_steps_iter=nve_run_time_steps_iter,
    nve_run_time_steps_lst=project_parameter['nve_run_time_steps_lst'],
    nve_run_time_steps=nve_run_time_steps,
    strain_result_lst=strain_result_lst,
    pressure_result_lst=pressure_result_lst,
    step_count=step_count,
    step_dict=step_dict,
    boundary_value=project_parameter['boundary_value'],
    ratio_boundary=project_parameter['ratio_boundary'],
    convergence_goal=project_parameter['convergence_criterion'],
    output_file=output_file
)
convergence_goal_achieved, enable_iteration, step_count, step_dict, timestep, fit_range, nve_run_time_steps, boundary_value, ratio_boundary, temperature_next, center = output

In [ ]:
step_dict[step_count]

In [ ]:
convergence_goal_achieved

# Step 4: full cycle, predict the final melting point

In [ ]:
temperature_estimate_lst, temperature_calculated_lst = [], []
while len(temperature_calculated_lst) < 3 or not convergence_goal_achieved and len(temperature_calculated_lst) < 10:
    temperature_estimate_lst.append(temperature_next)
    temperature_next, temperature_mean, temperature_left, temperature_right, strain_result_lst, pressure_result_lst = run_interface_iteration(
        basis=basis_minimize_vol.repeat([1,1,2]),
        temperature_next=temperature_next,
        project_parameter=project_parameter,
        project_path=project_path,
        timestep=timestep,
        nve_run_time_steps=nve_run_time_steps,
        fit_range=fit_range,
        center=center,
        debug_plot=True,
        lmp_command=lmp_command,
    )
    print(temperature_next, temperature_mean, temperature_left, temperature_right)
    output = validate_convergence(
        temperature_left=temperature_left,
        temperature_next=temperature_next,
        temperature_right=temperature_right,
        enable_iteration=enable_iteration,
        timestep_iter=timestep_iter,
        timestep_lst=project_parameter['timestep_lst'],
        timestep=timestep,
        fit_range_iter=fit_range_iter,
        fit_range_lst=project_parameter['fit_range_lst'],
        fit_range=fit_range,
        nve_run_time_steps_iter=nve_run_time_steps_iter,
        nve_run_time_steps_lst=project_parameter['nve_run_time_steps_lst'],
        nve_run_time_steps=nve_run_time_steps,
        strain_result_lst=strain_result_lst,
        pressure_result_lst=pressure_result_lst,
        step_count=step_count,
        step_dict=step_dict,
        boundary_value=project_parameter['boundary_value'],
        ratio_boundary=project_parameter['ratio_boundary'],
        convergence_goal=project_parameter['convergence_criterion'],
        output_file=output_file,
    )
    convergence_goal_achieved, enable_iteration, step_count, step_dict, timestep, fit_range, nve_run_time_steps, boundary_value, ratio_boundary, temperature_next, center = output
    print(step_dict[step_count])
    temperature_calculated_lst.append(temperature_next)

In [ ]:
if not os.path.exists(output_file):
    with open(output_file, 'w') as f:
        json.dump(step_dict, f)

In [ ]:
step_dict

In [ ]:
#plot the convergence of loop calculations
first_key = list(step_dict.keys())[-1]
second_key = 'temperature_next'
allt = [step_dict[key][second_key] for key in list(step_dict.keys())]
plt.plot(np.arange(1, len(allt)), allt[0:-1], 'ro-', label=r"$T^e$")
plt.plot(np.arange(1, len(allt)), allt[1:], 'bo-', label=r"$T^p$")
plt.legend(fontsize=14)
plt.tick_params(axis='both', labelsize=14)
plt.xlabel('Number of loops', fontsize=14)
plt.ylabel('Temperature (K)', fontsize=14)
plt.show()

In [ ]:
for k, v in sorted(step_dict.items()):
    print(k, v)